# 🚀 ADAN Trading Bot - Entraînement sur Colab

## Configuration Automatique Complète

Ce notebook configure et lance automatiquement l'entraînement du bot de trading ADAN sur Google Colab.

**⚠️ Important:**
- **Runtime**: CPU (pas besoin de GPU)
- **Durée estimée**: 1-2h pour 500k timesteps
- **Résultats**: Sauvegardés automatiquement dans Google Drive (optionnel)
- **Dépendances**: Toutes installées automatiquement

---

## 📋 Plan d'Exécution

1. **Cellule 1**: Installation et configuration (5-10 min)
2. **Cellule 2**: Connexion Google Drive (optionnel)
3. **Cellule 3**: Lancement de l'entraînement (1-2h)
4. **Cellule 4**: Monitoring en temps réel
5. **Cellule 5**: Sauvegarde des résultats
6. **Cellule 6**: Analyse des résultats

---

## 📦 Étape 1: Installation et Configuration (5-10 min)

Cette cellule:
- ✅ Installe toutes les dépendances système
- ✅ Installe tous les packages Python
- ✅ Clone le dépôt GitHub
- ✅ Vérifie les données
- ✅ Teste les imports

**Durée**: 5-10 minutes (première exécution)

In [ ]:
%%bash
# Configuration complète de l'environnement Colab
curl -sSL https://raw.githubusercontent.com/Cabrel10/ADAN0/main/setup_colab.sh | bash

## 💾 Étape 2: Connexion à Google Drive (Optionnel)

Pour sauvegarder automatiquement les résultats d'entraînement dans Google Drive.

**Optionnel**: Vous pouvez passer cette étape si vous ne voulez pas sauvegarder.

In [ ]:
from google.colab import drive
import os

# Monter Google Drive
drive.mount('/content/drive')

# Créer le dossier de sauvegarde
BACKUP_DIR = '/content/drive/MyDrive/ADAN_Training'
os.makedirs(BACKUP_DIR, exist_ok=True)

print(f"✅ Google Drive monté")
print(f"📁 Dossier de sauvegarde: {BACKUP_DIR}")

## 🚀 Étape 3: Lancement de l'Entraînement

### Options de Configuration:

| Timesteps | Durée | Recommandé pour |
|-----------|-------|------------------|
| **500,000** | 1-2h | Validation rapide |
| **1,000,000** | 2-4h | Test complet |
| **5,000,000** | 8-12h | Entraînement sérieux |
| **10,000,000** | 24-48h | Production (Colab Pro) |

**Par défaut**: 500,000 timesteps (validation rapide)

**Durée estimée**: 1-2 heures

In [ ]:
%%bash
cd /content/ADAN0

# Lancer l'entraînement avec 500k timesteps (défaut)
# Modifiez le nombre pour plus/moins de timesteps
bash launch_training.sh 500000

# Autres options:
# bash launch_training.sh 1000000    # 1M timesteps (2-4h)
# bash launch_training.sh 5000000    # 5M timesteps (8-12h)
# bash launch_training.sh 10000000   # 10M timesteps (24-48h, Colab Pro)

## 📊 Étape 4: Monitoring en Temps Réel

Exécutez cette cellule **pendant** l'entraînement pour voir les logs en temps réel.

**Appuyez sur le bouton Stop pour arrêter le monitoring** (l'entraînement continue).

In [ ]:
import subprocess
import time
import os
from IPython.display import clear_output, HTML
from datetime import datetime

print("📊 Monitoring des logs d'entraînement...")
print("(Appuyez sur le bouton Stop pour arrêter le monitoring)\n")

# Trouver le fichier de log le plus récent
log_dir = "/content/ADAN0/logs"
log_files = sorted([f for f in os.listdir(log_dir) if f.startswith('training_')], reverse=True)

if not log_files:
    print("❌ Aucun fichier de log trouvé")
    print("Assurez-vous que l'entraînement a démarré dans la cellule précédente")
else:
    log_file = os.path.join(log_dir, log_files[0])
    print(f"📝 Fichier de log: {log_files[0]}\n")
    print("=" * 80)
    
    last_line_count = 0
    
    try:
        while True:
            clear_output(wait=True)
            
            # Afficher l'heure de mise à jour
            print(f"🕐 Mise à jour: {datetime.now().strftime('%H:%M:%S')}")
            print(f"📝 Fichier: {log_files[0]}\n")
            print("=" * 80)
            
            # Afficher les dernières lignes pertinentes du log
            try:
                result = subprocess.run(
                    f"tail -50 {log_file} | grep -E 'DBE_DECISION|Episode|timestep|Portfolio|REWARD|Step' | tail -20",
                    shell=True,
                    capture_output=True,
                    text=True
                )
                if result.stdout:
                    print(result.stdout)
                else:
                    print("⏳ En attente des premiers logs...")
            except:
                print("⏳ En attente des logs...")
            
            print("=" * 80)
            
            # Afficher les statistiques
            try:
                dbe_count = subprocess.run(
                    f"grep -c 'DBE_DECISION' {log_file}",
                    shell=True,
                    capture_output=True,
                    text=True
                ).stdout.strip()
                
                regime_count = subprocess.run(
                    f"grep -c 'REGIME_DETECTION' {log_file}",
                    shell=True,
                    capture_output=True,
                    text=True
                ).stdout.strip()
                
                print(f"\n📈 Statistiques:")
                print(f"   - Décisions DBE: {dbe_count}")
                print(f"   - Détections régime: {regime_count}")
            except:
                pass
            
            print(f"\n⏳ Rafraîchissement dans 10 secondes... (Ctrl+C pour arrêter)")
            time.sleep(10)
            
    except KeyboardInterrupt:
        print("\n⏸️  Monitoring arrêté (l'entraînement continue)")

## 💾 Étape 5: Sauvegarde des Résultats sur Google Drive

Exécutez cette cellule **après** l'entraînement pour sauvegarder les résultats.

Cela sauvegarde:
- ✅ Checkpoints du modèle
- ✅ Logs d'entraînement
- ✅ Résultats et métriques

In [ ]:
import shutil
import os
from datetime import datetime

# Créer un dossier avec timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = f"/content/drive/MyDrive/ADAN_Training/run_{timestamp}"
os.makedirs(backup_path, exist_ok=True)

print("💾 Sauvegarde des résultats...\n")

# Copier les checkpoints
if os.path.exists("/content/ADAN0/checkpoints"):
    try:
        shutil.copytree("/content/ADAN0/checkpoints", 
                        f"{backup_path}/checkpoints", 
                        dirs_exist_ok=True)
        print("✅ Checkpoints sauvegardés")
    except Exception as e:
        print(f"⚠️  Erreur lors de la sauvegarde des checkpoints: {e}")

# Copier les logs
if os.path.exists("/content/ADAN0/logs"):
    try:
        shutil.copytree("/content/ADAN0/logs", 
                        f"{backup_path}/logs", 
                        dirs_exist_ok=True)
        print("✅ Logs sauvegardés")
    except Exception as e:
        print(f"⚠️  Erreur lors de la sauvegarde des logs: {e}")

# Copier les résultats
if os.path.exists("/content/ADAN0/results"):
    try:
        shutil.copytree("/content/ADAN0/results", 
                        f"{backup_path}/results", 
                        dirs_exist_ok=True)
        print("✅ Résultats sauvegardés")
    except Exception as e:
        print(f"⚠️  Erreur lors de la sauvegarde des résultats: {e}")

print(f"\n✅ Sauvegarde complète!")
print(f"📁 Emplacement: {backup_path}")
print(f"\n💡 Vous pouvez maintenant télécharger les résultats depuis Google Drive")

## 📊 Étape 6: Analyse Rapide des Résultats

Affiche les statistiques d'entraînement et les métriques clés.

In [ ]:
import subprocess
import os
from datetime import datetime

# Trouver le fichier de log le plus récent
log_dir = "/content/ADAN0/logs"
log_files = sorted([f for f in os.listdir(log_dir) if f.startswith('training_')], reverse=True)

if not log_files:
    print("❌ Aucun fichier de log trouvé")
else:
    log_file = os.path.join(log_dir, log_files[0])
    
    print("📊 Analyse des résultats d'entraînement\n")
    print("=" * 80)
    
    # Compter les décisions par worker
    workers = {
        'Trial26 Ultra-Stable': 'w0',
        'Moderate Optimized': 'w1',
        'Sharpe Optimized': 'w2',
        'Aggressive Optimized': 'w3'
    }
    
    print("\n🤖 Décisions par Worker:\n")
    for worker_name, worker_id in workers.items():
        result = subprocess.run(
            f"grep -c '{worker_name}' {log_file}",
            shell=True,
            capture_output=True,
            text=True
        )
        count = result.stdout.strip()
        print(f"   ✅ {worker_name:25s} ({worker_id}): {count:>6s} décisions")
    
    print("\n" + "=" * 80)
    
    # Statistiques globales
    print("\n📈 Statistiques Globales:\n")
    
    dbe_result = subprocess.run(
        f"grep -c 'DBE_DECISION' {log_file}",
        shell=True,
        capture_output=True,
        text=True
    )
    dbe_count = dbe_result.stdout.strip()
    
    regime_result = subprocess.run(
        f"grep -c 'REGIME_DETECTION' {log_file}",
        shell=True,
        capture_output=True,
        text=True
    )
    regime_count = regime_result.stdout.strip()
    
    lines_result = subprocess.run(
        f"wc -l < {log_file}",
        shell=True,
        capture_output=True,
        text=True
    )
    lines_count = lines_result.stdout.strip()
    
    print(f"   - Décisions DBE: {dbe_count}")
    print(f"   - Détections régime: {regime_count}")
    print(f"   - Lignes de log: {lines_count}")
    
    # Erreurs
    error_result = subprocess.run(
        f"grep -i 'error\\|exception' {log_file} | wc -l",
        shell=True,
        capture_output=True,
        text=True
    )
    error_count = error_result.stdout.strip()
    
    print(f"   - Erreurs: {error_count}")
    
    print("\n" + "=" * 80)
    
    # Dernières lignes du log
    print("\n📋 Dernières lignes du log:\n")
    
    tail_result = subprocess.run(
        f"tail -10 {log_file}",
        shell=True,
        capture_output=True,
        text=True
    )
    
    for line in tail_result.stdout.split('\n')[-10:]:
        if line:
            print(f"   {line}")
    
    print("\n" + "=" * 80)
    print("\n✅ Analyse terminée!")

## 🔄 Étape 7: Relancer avec Plus de Timesteps (Optionnel)

Pour un entraînement plus complet avec plus de timesteps.

**⚠️ Attention**: Nécessite Colab Pro pour les durées > 12h

In [ ]:
%%bash
cd /content/ADAN0

# Décommenter la ligne souhaitée:

# 1M timesteps (2-4 heures)
# bash launch_training.sh 1000000

# 5M timesteps (8-12 heures)
# bash launch_training.sh 5000000

# 10M timesteps (24-48 heures - Colab Pro requis)
# bash launch_training.sh 10000000

echo "Décommenter la ligne souhaitée et exécuter la cellule"

---

## 📚 Documentation Complète

### Fichiers Disponibles
- **setup_colab.sh**: Script d'installation automatique
- **launch_training.sh**: Script de lancement d'entraînement
- **config/config.yaml**: Configuration du modèle
- **scripts/train_parallel_agents.py**: Script d'entraînement principal

### Dépendances Installées
- PyTorch 2.1.0 (CPU)
- Stable-Baselines3 2.1.0
- Optuna 3.4.0
- Pandas, NumPy, Scikit-learn
- TA-Lib (indicateurs techniques)

### Support
- **Repository**: https://github.com/Cabrel10/ADAN0
- **Issues**: Consultez les logs pour les erreurs
- **Monitoring**: Utilisez la cellule de monitoring en temps réel

### Durées Estimées
| Timesteps | Durée | Colab | Colab Pro |
|-----------|-------|-------|----------|
| 500k | 1-2h | ✅ | ✅ |
| 1M | 2-4h | ✅ | ✅ |
| 5M | 8-12h | ⚠️ | ✅ |
| 10M | 24-48h | ❌ | ✅ |

---

**Créé avec ❤️ pour ADAN Trading Bot**